In [10]:
# =====================
# Load Libraries
# =====================

import pandas as pd
import requests
import re
from io import StringIO

# =====================
#Load data from Wikipedia
# =====================

url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

tables = pd.read_html(StringIO(html))

print("Tables loaded:", len(tables))
standing_tables = []

for i, table in enumerate(tables):
    cols = list(table.columns)

    if any("Team" in str(c) for c in cols) and any("Pts" in str(c) for c in cols):
        standing_tables.append(i)

# =====================
# Build Group Information
# =====================

standing_tables = standing_tables[:12]

groups = {}

for group, table_idx in zip(list("ABCDEFGHIJKL"), standing_tables):
    table = tables[table_idx]

    team_col = [c for c in table.columns if "Team" in str(c)][0]

    teams = (
        table[team_col]
        .astype(str)
        .str.replace(r"\s*\(.*?\)", "", regex=True)
        .str.strip()
        .tolist()
    )

    groups[group] = teams

# =====================
# Extract Match Results
# =====================

matches = []

match_info = {}

for i, table in enumerate(tables):
    if table.shape[1] < 3:
        continue

    team1 = str(table.columns[0]).strip()
    score = str(table.columns[1]).strip()
    team2 = str(table.columns[2]).strip()

    if not re.match(r"^\d+\s*[–-]\s*\d+$", score):
        continue

    s1, s2 = re.split(r"[–-]", score)
    s1 = int(s1.strip())
    s2 = int(s2.strip())

    group_found = None

    for g, team_list in groups.items():
        if team1 in team_list and team2 in team_list:
            group_found = g
            break

    if group_found is not None:
        matches.append((group_found, team1, s1, team2, s2))
        match_key = tuple(sorted([team1, team2]))
        match_info[match_key] = {
            "Group": group_found,
            "Score": f"{s1}-{s2}",
            "Status": "Played"
        }

for i, table in enumerate(tables):
    if table.shape[1] < 3:
        continue

    team1 = str(table.columns[0]).strip()
    middle = str(table.columns[1]).strip()
    team2 = str(table.columns[2]).strip()

    if not middle.startswith("Match"):
        continue

    group_found = None

    for g, team_list in groups.items():
        if team1 in team_list and team2 in team_list:
            group_found = g
            break

    if group_found is not None:
        match_key = tuple(sorted([team1, team2]))
        match_info[match_key] = {
            "Group": group_found,
            "Score": "⏳",
            "Status": "Remaining"
        }

# =====================
# Dashboard Functions
# =====================

from itertools import product

def calculate_group_status(group, table, matrix):
    teams = list(table.index)

    remaining_games = []

    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            t1 = teams[i]
            t2 = teams[j]

            if matrix.loc[t1, t2] == "⏳":
                remaining_games.append((t1, t2))

    possible_top2 = {team: False for team in teams}
    always_top2 = {team: True for team in teams}

    possible_scores = [
    (0,0),
    (1,0), (2,0), (3,0), (4,0), (5,0),
    (0,1), (0,2), (0,3), (0,4), (0,5),
    (1,1), (2,2), (3,3),
    (2,1), (3,1), (3,2),
    (1,2), (1,3), (2,3),
    ]

    outcomes = list(product(possible_scores, repeat=len(remaining_games)))

    for outcome_set in outcomes:
        sim = table[["MP","W","D","L","GF","GA","GD","Pts"]].copy()

        for (t1, t2), (s1, s2) in zip(remaining_games, outcome_set):

            sim.loc[t1, "MP"] += 1
            sim.loc[t2, "MP"] += 1

            sim.loc[t1, "GF"] += s1
            sim.loc[t1, "GA"] += s2
            sim.loc[t2, "GF"] += s2
            sim.loc[t2, "GA"] += s1

            if s1 > s2:
                sim.loc[t1, "W"] += 1
                sim.loc[t2, "L"] += 1
                sim.loc[t1, "Pts"] += 3

            elif s1 < s2:
                sim.loc[t2, "W"] += 1
                sim.loc[t1, "L"] += 1
                sim.loc[t2, "Pts"] += 3

            else:
                sim.loc[t1, "D"] += 1
                sim.loc[t2, "D"] += 1
                sim.loc[t1, "Pts"] += 1
                sim.loc[t2, "Pts"] += 1

        sim["GD"] = sim["GF"] - sim["GA"]
        sim = sim.sort_values(
            ["Pts", "GD", "GF"],
            ascending=False
        )
        top2 = set(sim.index[:2])

        for team in teams:
            if team in top2:
                possible_top2[team] = True
            else:
                always_top2[team] = False

    statuses = []

    for team in table.index:
        if always_top2[team]:
            statuses.append("✓ Qualified")
        elif not possible_top2[team]:
            statuses.append("✗ Eliminated")
        else:
            statuses.append("○ Alive")

    return statuses

def show_group(group):
    teams = groups[group]

# =====================
# Group Standings
# =====================
    table = pd.DataFrame(
        0,
        index=teams,
        columns=["MP","W","D","L","GF","GA","GD","Pts"]
    )

    for g, team1, score1, team2, score2 in matches:
        if g != group:
            continue

        table.loc[team1, "MP"] += 1
        table.loc[team2, "MP"] += 1

        table.loc[team1, "GF"] += score1
        table.loc[team1, "GA"] += score2
        table.loc[team2, "GF"] += score2
        table.loc[team2, "GA"] += score1

        if score1 > score2:
            table.loc[team1, "W"] += 1
            table.loc[team2, "L"] += 1
            table.loc[team1, "Pts"] += 3

        elif score1 < score2:
            table.loc[team2, "W"] += 1
            table.loc[team1, "L"] += 1
            table.loc[team2, "Pts"] += 3

        else:
            table.loc[team1, "D"] += 1
            table.loc[team2, "D"] += 1
            table.loc[team1, "Pts"] += 1
            table.loc[team2, "Pts"] += 1

    table["GD"] = table["GF"] - table["GA"]

    table = table.sort_values(
        ["Pts", "GD", "GF"],
        ascending=False
    )


    # =====================
    # Match Matrix
    # =====================
    matrix = pd.DataFrame(
        "⏳",
        index=teams,
        columns=teams
    )

    for team in teams:
        matrix.loc[team, team] = "—"

    for g, team1, score1, team2, score2 in matches:
        if g != group:
            continue

        matrix.loc[team1, team2] = f"{score1}-{score2}"
        matrix.loc[team2, team1] = f"{score2}-{score1}"

    table["Status"] = calculate_group_status(group, table, matrix)

    print(f"Group {group} Standings")
    display(table)

    print(f"Group {group} Match Matrix")
    display(matrix)

    # =====================
    # Remaining Matches
    # =====================
    remaining = []

    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            t1 = teams[i]
            t2 = teams[j]

            if matrix.loc[t1, t2] == "⏳":
                match_key = tuple(sorted([t1, t2]))
                info = match_info.get(match_key, {})

                remaining.append({
                    "Match": f"{t1} vs {t2}",
                })

    print("Remaining Matches")
    remaining_df = pd.DataFrame(remaining)
    display(remaining_df.style.hide(axis="index"))

def show_all_groups():

    print("已讀取比賽數：", len(matches))
    print("小組：", sorted(groups.keys()))

    for group in sorted(groups.keys()):

        print("\n" + "="*60)
        print(f"GROUP {group}")
        print("="*60)

        show_group(group)

print("⚽ World Cup Dashboard Ready")
print(f"Matches loaded: {len(matches)}")

# =====================
# Dashboard UI
# =====================

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

title = HTML("""
<h2>🌎 World Cup Group Dashboard</h2>
<p>Choose a group to view standings, match matrix, and remaining matches.</p>
""")

group_selector = widgets.Dropdown(
    options=sorted(groups.keys()),
    value="A",
    description="Group:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px")
)

refresh_button = widgets.Button(
    description="Refresh",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(width="120px")
)

output = widgets.Output()

def render_dashboard(group):
    with output:
        clear_output(wait=True)
        show_group(group)

def on_group_change(change):
    if change["name"] == "value":
        render_dashboard(change["new"])

def on_refresh_click(b):
    render_dashboard(group_selector.value)

group_selector.observe(on_group_change, names="value")
refresh_button.on_click(on_refresh_click)

# =====================
# Launch Dashboard
# =====================

display(title)
display(widgets.HBox([group_selector, refresh_button]))
display(output)

render_dashboard(group_selector.value)


Tables loaded: 150
⚽ World Cup Dashboard Ready
Matches loaded: 47


Output()